In [1]:
import numpy as np
import tensorflow as tf
from scipy import signal
from tensorflow.keras import layers, Model, initializers
import matplotlib.pyplot as plt

from optic.models.devices import mzm, photodiode
from optic.models.channels import linearFiberChannel
from optic.comm.sources import bitSource
from optic.comm.modulation import modulateGray
from optic.comm.metrics import bert
from optic.dsp.core import firFilter, pulseShape, upsample, anorm
from optic.utils import parameters, dBm2W

# ====== Simulation Parameters ======
SpS = 16  # samples per symbol
M = 2  # order of the modulation format (PAM-2)
Rs = 10e9  # Symbol rate
Fs = SpS * Rs  # Signal sampling frequency (samples/second)
Pi_dBm = 3  # laser optical power at the input of the MZM in dBm
Pi = dBm2W(Pi_dBm)  # convert from dBm to W

paramBits = parameters()
paramBits.nBits = 2**18
paramBits.mode = 'random'
paramBits.seed = 123 

paramPulse = parameters()
paramPulse.pulseType = 'nrz'
paramPulse.SpS = SpS     

paramMZM = parameters()
paramMZM.Vpi = 2
paramMZM.Vb = -paramMZM.Vpi / 2

paramCh = parameters()
paramCh.L = 100
paramCh.alpha = 0.2
paramCh.D = 16
paramCh.Fc = 193.1e12
paramCh.Fs = Fs

paramPD = parameters()
paramPD.ideal = False
paramPD.B = Rs
paramPD.Fs = Fs
paramPD.seed = 456

def optical_chain(symbTx):
    symbolsUp = upsample(symbTx, SpS)
    pulse = pulseShape(paramPulse)
    sigTx = firFilter(pulse, symbolsUp)
    sigTx = anorm(sigTx) # normalize to 1 Vpp

    # Artifical distortion (non linear)
    alpha = 0.5
    sigTx = sigTx + alpha * sigTx**3

    Ai = np.sqrt(Pi) 
    sigTxo = mzm(Ai, sigTx, paramMZM)
    sigCh = linearFiberChannel(sigTxo, paramCh)
    I_Rx = photodiode(sigCh, paramPD)
    
    # capture samples in the middle of signaling intervals
    I_Rx = I_Rx[0::SpS]
    return I_Rx

In [2]:
bitsTx = bitSource(paramBits)
symbTx = modulateGray(bitsTx, M, "pam")

# Run Baseline
print("Running Baseline Simulation (Before DPD)...")
I_Rx_baseline = optical_chain(symbTx)
BER_baseline, Q_baseline = bert(I_Rx_baseline, bitsTx)

print("\n>>> Baseline Locked (Before DPD) <<<")
print(f"BER: {BER_baseline:.3e} | Q-factor: {Q_baseline:.2f}")

Running Baseline Simulation (Before DPD)...

>>> Baseline Locked (Before DPD) <<<
BER: 3.311e-03 | Q-factor: 2.59


In [3]:
#Wiener-Hammerstein (WH)，Based on scipy FIR
class PureWienerHammersteinDPD:
    def __init__(self, mem1=11, mem2=11, poly_orders=(1, 3, 5, 7)):
        self.mem1 = mem1
        self.mem2 = mem2
        self.poly_orders = poly_orders
        
        # w1(the first fir fliter), c(polynomial order), w2(the second fir filter)
        w1_init = np.zeros((mem1, 1), dtype=np.float64)
        w1_init[mem1 // 2, 0] = 1.0
        self.w1 = tf.Variable(w1_init, dtype=tf.float64)
        
        c_init = np.zeros((len(poly_orders), 1), dtype=np.float64)
        c_init[0, 0] = 1.0
        self.c = tf.Variable(c_init, dtype=tf.float64)
        
        w2_init = np.zeros((mem2, 1), dtype=np.float64)
        w2_init[mem2 // 2, 0] = 1.0
        self.w2 = tf.Variable(w2_init, dtype=tf.float64)

    def apply_dpd(self, x):
        """
        Forward pre-distortion: Implementing the WH formula using classical signal processing (scipy.signal.lfilter)
        """
        x = np.asarray(x, dtype=np.float64).ravel()
        w1_np = self.w1.numpy().flatten()
        c_np = self.c.numpy().flatten()
        w2_np = self.w2.numpy().flatten()
        
        # 1. Wiener FIR Filter
        v = signal.lfilter(w1_np, [1.0], x)
        
        # 2. Nonlinear Polynomial Distortion Compensation
        z = np.zeros_like(v, dtype=np.float64)
        for i, p in enumerate(self.poly_orders):
            z += c_np[i] * (v ** p)
            
        # 3. Hammerstein FIR Filter
        y = signal.lfilter(w2_np, [1.0], z)
        return y

    def train_step(self, x_in, y_target, lr=0.01, epochs=40):
        """
        Training: Using TensorFlow's automatic differentiation to find the optimal solution to the Wiener-Hammerstein equation
        """
        N = len(x_in)
        T_matrix = np.zeros((N, self.mem1), dtype=np.float64)
        for m in range(self.mem1):
            T_matrix[m:, m] = x_in[:N-m]
        X1_tf = tf.constant(T_matrix, dtype=tf.float64)
        y_target_tf = tf.constant(y_target[:, None], dtype=tf.float64)
        
        optimizer = tf.keras.optimizers.Adam(learning_rate=lr)
        
        for epoch in range(epochs):
            with tf.GradientTape() as tape:
                v = tf.matmul(X1_tf, self.w1)
                
                z = tf.zeros_like(v, dtype=tf.float64)
                for i, p in enumerate(self.poly_orders):
                    z += self.c[i, 0] * tf.pow(v, float(p))
                    
                z_shifts = []
                for m in range(self.mem2):
                    if m == 0:
                        z_shifts.append(z)
                    else:
                        z_shift = tf.concat([tf.zeros([m, 1], dtype=tf.float64), z[:-m]], axis=0)
                        z_shifts.append(z_shift)
                Z2 = tf.concat(z_shifts, axis=1) 
                
                y_pred = tf.matmul(Z2, self.w2)
                loss = tf.reduce_mean(tf.square(y_target_tf - y_pred))
                
            grads = tape.gradient(loss, [self.w1, self.c, self.w2])
            optimizer.apply_gradients(zip(grads, [self.w1, self.c, self.w2]))

In [4]:
# Signal Alignment Tool
def align_and_scale(ref, rx):
    """
    Determine the delay of the received signal using cross-correlation and align it precisely to the nanosecond level.
    Simultaneously perform amplitude calibration to prevent model collapse caused by system gain drift.
    """
    ref_norm = ref - np.mean(ref)
    rx_norm = rx - np.mean(rx)
    cc = np.correlate(ref_norm, rx_norm, mode='full')
    delay = np.argmax(np.abs(cc)) - (len(rx) - 1)
    
    if delay > 0:
        rx_aligned = np.pad(rx, (delay, 0), mode='constant')[:len(ref)]
    elif delay < 0:
        rx_aligned = np.pad(rx[-delay:], (0, max(0, len(ref) - (len(rx)+delay))), mode='constant')[:len(ref)]
    else:
        rx_aligned = rx.copy()
        
    gain = np.sum(ref * rx_aligned) / (np.sum(rx_aligned**2) + 1e-12)
    return rx_aligned * gain

In [ ]:
# Execute the Damped ILA training loop
main_dpd = PureWienerHammersteinDPD(mem1=11, mem2=11, poly_orders=(1, 3, 5, 7))

ila_iterations = 15
alpha_damping = 0.25  # Damping coefficient, ensuring stable convergence of the BER
best_ber = float('inf')
best_weights = None

print("Iteration | Q-factor  | BER")
print("-" * 35)

for iteration in range(ila_iterations):
    # 1. Forward pre-distortion (using pure mathematical formulas)
    if iteration == 0:
        z_dpd = symbTx.copy()
    else:
        z_dpd = main_dpd.apply_dpd(symbTx)
        z_dpd = z_dpd * (np.std(symbTx) / (np.std(z_dpd) + 1e-12)) # Power alignment

    # 2. Passed through the channel and evaluated
    I_Rx = optical_chain(z_dpd)
    BER, Q = bert(I_Rx, bitsTx)
    print(f"{iteration+1:<9} | {Q:<10.2f} | {BER:<10.2e}")

    # 3. Record your best set of scores
    if BER < best_ber: 
        best_weights = [main_dpd.w1.numpy(), main_dpd.c.numpy(), main_dpd.w2.numpy()]
        best_ber = BER

    # 4. Obtain a perfectly aligned received signal
    y_aligned = align_and_scale(z_dpd, I_Rx)

    # 5. Train a reverse model (distortion after extraction)
    inv_model = PureWienerHammersteinDPD(mem1=11, mem2=11, poly_orders=(1, 3, 5, 7))
    inv_model.train_step(x_in=y_aligned, y_target=z_dpd, lr=0.01, epochs=40)

    # 6. Update damping for launch targets
    ideal_z = inv_model.apply_dpd(symbTx)
    next_target = symbTx + alpha_damping * (ideal_z - symbTx)
    next_target = next_target * (np.std(symbTx) / (np.std(next_target) + 1e-12))
    
    # 7. Self-learning of the main model
    main_dpd.train_step(x_in=symbTx, y_target=next_target, lr=0.01, epochs=30)

print("\nTraining Complete.")

Iteration | Q-factor  | BER
-----------------------------------


2026-03-17 22:22:39.589520: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3
2026-03-17 22:22:39.589538: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 24.00 GB
2026-03-17 22:22:39.589544: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 8.88 GB
2026-03-17 22:22:39.589559: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-03-17 22:22:39.589567: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


1         | 2.59       | 3.31e-03  


In [ ]:
# Comparison
from optic.comm.sources import bitSource
from optic.comm.modulation import modulateGray
from optic.comm.metrics import bert, fastBERcalc, calcEVM

# 1. Use a new random seed to generate data that hasn't been seen before
paramBits.seed = 3455
bitsTx_test = bitSource(paramBits)
symbTx_test = modulateGray(bitsTx_test, M, "pam")

# Before DPD

I_Rx_no_dpd = optical_chain(symbTx_test)
BER_test_base, Q_test_base = bert(I_Rx_no_dpd, bitsTx_test)

# Strictly aligned signals for calculating detailed metrics
rx_base_aligned = align_and_scale(symbTx_test, I_Rx_no_dpd)
_, SER_base, SNR_base = fastBERcalc(rx_base_aligned, symbTx_test, M, 'pam')
EVM_base = calcEVM(rx_base_aligned, M, 'pam', symbTx_test)

# Extract values (the optic library typically returns a single-element list or array)
SER_b = float(SER_base[0])
SNR_b = float(SNR_base[0])
EVM_b = float(EVM_base[0] * 100) 

#After DPD

# Load the optimal parameters for the WH model
main_dpd.w1.assign(best_weights[0])
main_dpd.c.assign(best_weights[1])
main_dpd.w2.assign(best_weights[2])

# excute WH DPD
z_signal_dpd = main_dpd.apply_dpd(symbTx_test)
z_signal_dpd = z_signal_dpd * (np.std(symbTx_test) / (np.std(z_signal_dpd) + 1e-12))

# over a fiber-optic link
I_Rx_dpd = optical_chain(z_signal_dpd)
BER_test_wh, Q_test_wh = bert(I_Rx_dpd, bitsTx_test)

# Strictly aligned signals for calculating detailed metrics
rx_wh_aligned = align_and_scale(symbTx_test, I_Rx_dpd)
_, SER_wh, SNR_wh = fastBERcalc(rx_wh_aligned, symbTx_test, M, 'pam')
EVM_wh = calcEVM(rx_wh_aligned, M, 'pam', symbTx_test)


SER_w = float(SER_wh[0])
SNR_w = float(SNR_wh[0])
EVM_w = float(EVM_wh[0] * 100)

# Print the Ultimate Multi-Metric Comparison Chart

print("\n" + "="*75)
print(" FINAL TEST SET PERFORMANCE: NO DPD vs. After WH-DPD")
print("="*75)
print(f"{'Metric':<15} | {'Before DPD':<15} | {'After WH-DPD':<15} | {'Delta (Improvement)'}")
print("-" * 75)

# Print the 5 Key Metrics
print(f"{'BER':<15} | {BER_test_base:<15.3e} | {BER_test_wh:<15.3e} | {BER_test_wh - BER_test_base:+.3e}")
print(f"{'SER':<15} | {SER_b:<15.3e} | {SER_w:<15.3e} | {SER_w - SER_b:+.3e}")
print(f"{'EVM (%)':<15} | {EVM_b:<15.3f} | {EVM_w:<15.3f} | {EVM_w - EVM_b:+.3f}")
print(f"{'SNR (dB)':<15} | {SNR_b:<15.3f} | {SNR_w:<15.3f} | {SNR_w - SNR_b:+.3f}")
print(f"{'Q-factor':<15} | {Q_test_base:<15.3f} | {Q_test_wh:<15.3f} | {Q_test_wh - Q_test_base:+.3f}")
print("-" * 75)
print("Model Core: SciPy Signal Processing")


 FINAL TEST SET PERFORMANCE: NO DPD vs. After WH-DPD
Metric          | Before DPD      | After WH-DPD    | Delta (Improvement)
---------------------------------------------------------------------------
BER             | 3.410e-03       | 1.869e-04       | -3.223e-03
SER             | 4.132e-01       | 3.910e-01       | -2.228e-02
EVM (%)         | 81.608          | 77.795          | -3.813
SNR (dB)        | 0.883           | 1.090           | +0.208
Q-factor        | 2.581           | 3.500           | +0.919
---------------------------------------------------------------------------
Model Core: Pure Math & SciPy Signal Processing
